# Identify Unexpectedly Affected Households

This notebook identifies households that **should not** have been affected by the transit scenario but **were**. 

The transit scenario improves transit service in specific zones (defined in `transit_zones_affected.csv`). Households outside these zones should theoretically show no differences between base and transit runs. This notebook finds households that violate this expectation, which can then be investigated using ActivitySim trace mode.

## 1. Import Libraries and Define Paths

Import required libraries and define file paths for the base and transit output directories, as well as the transit zones CSV.

In [1]:
import pandas as pd
from pathlib import Path

# Define paths
PROJECT_ROOT = Path("..").resolve()
BASE_DIR = PROJECT_ROOT / "output-base-eet"
TRANSIT_DIR = PROJECT_ROOT / "output-transit-eet"
TRANSIT_ZONES_CSV = PROJECT_ROOT / "input" / "transit_zones_affected.csv"

# Output file for unexpectedly affected households
OUTPUT_CSV = PROJECT_ROOT / "input" / "unexpectedly_affected_households.csv"

print(f"Base directory: {BASE_DIR}")
print(f"Transit directory: {TRANSIT_DIR}")
print(f"Transit zones CSV: {TRANSIT_ZONES_CSV}")

Base directory: /Users/tomstephen/dev/asim_eet_viz/output-base-eet
Transit directory: /Users/tomstephen/dev/asim_eet_viz/output-transit-eet
Transit zones CSV: /Users/tomstephen/dev/asim_eet_viz/input/transit_zones_affected.csv


## 2. Load Affected Transit Zones

Read the transit zones CSV to get the list of MAZ/TAZ that were targeted for transit improvements.

In [2]:
# Load transit zones
transit_zones = pd.read_csv(TRANSIT_ZONES_CSV)
affected_maz = set(transit_zones["MAZ"].unique())
affected_taz = set(transit_zones["taz"].unique())

print(f"Number of affected MAZ: {len(affected_maz)}")
print(f"Number of affected TAZ: {len(affected_taz)}")
print(f"\nSample affected MAZ: {sorted(list(affected_maz))[:10]}")

Number of affected MAZ: 2365
Number of affected TAZ: 676

Sample affected MAZ: [np.int64(2), np.int64(12), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(25), np.int64(28), np.int64(36)]


## 3. Load Household and Person Data from Base and Transit Outputs

Load the household and person parquet files from both scenarios. Non-mandatory tour frequency is a person-level attribute.

In [3]:
# Load household data from both scenarios
base_hh = pd.read_parquet(BASE_DIR / "final_households.parquet")
transit_hh = pd.read_parquet(TRANSIT_DIR / "final_households.parquet")

# Load person data (non_mandatory_tour_frequency is person-level)
base_persons = pd.read_parquet(BASE_DIR / "final_persons.parquet")
transit_persons = pd.read_parquet(TRANSIT_DIR / "final_persons.parquet")

print(f"Base households: {len(base_hh)}, Transit households: {len(transit_hh)}")
print(f"Base persons: {len(base_persons)}, Transit persons: {len(transit_persons)}")
print(f"\nPerson columns related to tours:")
tour_cols = [c for c in base_persons.columns if "tour" in c.lower() or "non_mandatory" in c.lower()]
print(tour_cols)

Base households: 1276883, Transit households: 1276883
Base persons: 3282455, Transit persons: 3282455

Person columns related to tours:
['mandatory_tour_frequency', 'num_work_tours', 'num_joint_tours', 'non_mandatory_tour_frequency', 'num_escort_tours', 'num_eatout_tours', 'num_shop_tours', 'num_maint_tours', 'num_discr_tours', 'num_social_tours', 'num_non_escort_tours', 'num_shop_maint_tours', 'num_shop_maint_escort_tours', 'num_add_shop_maint_tours', 'num_soc_discr_tours', 'num_add_soc_discr_tours']


## 4. Identify Persons Outside Affected Zones

Filter to persons whose household's `home_zone_id` (MAZ) is NOT in the affected transit zones list. These are the "control group" that should remain unchanged.

In [4]:
# Get list of unaffected household IDs
# household_id is the INDEX, not a column
unaffected_hh_ids = set(base_hh[~base_hh["home_zone_id"].isin(affected_maz)].index)

# Filter persons to those in unaffected households
# household_id is also the index in persons parquet
base_persons_unaffected = base_persons[base_persons.index.isin(unaffected_hh_ids)].copy()
transit_persons_unaffected = transit_persons[transit_persons.index.isin(unaffected_hh_ids)].copy()

print(f"Base persons outside affected zones: {len(base_persons_unaffected)}")
print(f"Transit persons outside affected zones: {len(transit_persons_unaffected)}")
print(f"\nTotal persons in affected zones (base): {len(base_persons) - len(base_persons_unaffected)}")
print(f"Total persons in affected zones (transit): {len(transit_persons) - len(transit_persons_unaffected)}")

# Check non_mandatory_tour_frequency distribution
print("\nNon-mandatory tour frequency distribution (base, unaffected persons):")
print(base_persons_unaffected["non_mandatory_tour_frequency"].value_counts().sort_index())

Base persons outside affected zones: 1104343
Transit persons outside affected zones: 1104343

Total persons in affected zones (base): 2178112
Total persons in affected zones (transit): 2178112

Non-mandatory tour frequency distribution (base, unaffected persons):
non_mandatory_tour_frequency
0      644073
1       87816
2       13466
3        9867
4        1286
        ...  
192       104
193        11
194         5
195         1
196        12
Name: count, Length: 189, dtype: int64


## 5. Compare Non-Mandatory Tour Frequency for Unaffected Persons

Merge the filtered base and transit person DataFrames on `person_id` and compare `non_mandatory_tour_frequency` to find differences.

In [9]:
# Merge base and transit on person_id for unaffected persons
# person_id might be the index, so reset if needed
base_p = base_persons_unaffected.reset_index() if "person_id" not in base_persons_unaffected.columns else base_persons_unaffected
transit_p = transit_persons_unaffected.reset_index() if "person_id" not in transit_persons_unaffected.columns else transit_persons_unaffected

merged = base_p.merge(
    transit_p,
    on="person_id",
    suffixes=("_base", "_transit"),
    how="inner"
)

print(f"Persons present in both base and transit (outside affected zones): {len(merged)}")

# Compare non_mandatory_tour_frequency
diff_mask = merged["non_mandatory_tour_frequency_base"] != merged["non_mandatory_tour_frequency_transit"]
unexpectedly_affected = merged[diff_mask].copy()

print("Merged head")
my_df = merged[["person_id", "non_mandatory_tour_frequency_base", "non_mandatory_tour_frequency_transit"]]
my_df["changed"] = my_df["non_mandatory_tour_frequency_base"] != my_df["non_mandatory_tour_frequency_transit"]
changed_only = my_df[my_df["changed"]]
display(changed_only.head())

print(f"\n*** Persons outside affected zones with different non-mandatory tour frequency: {len(unexpectedly_affected)} ***")

# Show the change patterns
if len(unexpectedly_affected) > 0:
    print("\nChange patterns (base -> transit):")
    change_summary = pd.crosstab(
        unexpectedly_affected["non_mandatory_tour_frequency_base"],
        unexpectedly_affected["non_mandatory_tour_frequency_transit"],
        rownames=["Base"],
        colnames=["Transit"]
    )
    display(change_summary)

# Also find persons that exist in one but not the other
base_person_ids = set(base_p["person_id"])
transit_person_ids = set(transit_p["person_id"])
base_only = base_person_ids - transit_person_ids
transit_only = transit_person_ids - base_person_ids

print(f"\nPersons only in base (outside affected zones): {len(base_only)}")
print(f"Persons only in transit (outside affected zones): {len(transit_only)}")

Persons present in both base and transit (outside affected zones): 1104343
Merged head


/var/folders/gq/dgbn70813y75d07_cpnh7b6h0000gn/T/ipykernel_38315/1616557531.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  my_df["changed"] = my_df["non_mandatory_tour_frequency_base"] != my_df["non_mandatory_tour_frequency_transit"]


,person_id,non_mandatory_tour_frequency_base,non_mandatory_tour_frequency_transit,changed
1190,1191,120,35,True
4326,4327,12,89,True
4328,4329,1,0,True
6849,6850,156,89,True
13318,13319,6,89,True



*** Persons outside affected zones with different non-mandatory tour frequency: 129 ***

Change patterns (base -> transit):


Transit,0,1,2,4,6,12,13,15,24,25,...,120,121,156,157,158,160,167,168,175,180
Base,,,,,,,,,,,,,,,,,,,,,
0,0,5,0,0,1,2,0,0,0,0,...,0,0,4,2,0,0,0,0,0,1
1,5,0,0,0,0,0,0,0,0,0,...,1,0,2,1,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12,1,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
13,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
20,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



Persons only in base (outside affected zones): 0
Persons only in transit (outside affected zones): 0


In [6]:
# list some of the affected households
if len(unexpectedly_affected) > 0:
    affected_hh_ids = set(unexpectedly_affected["household_id_base"].unique())
    affected_hh_df = base_hh[base_hh.index.isin(affected_hh_ids)].copy()
    print(f"\nSample of unexpectedly affected households (home_zone_id, income, hhsize, auto_ownership):")
    display(affected_hh_df[["home_zone_id", "income", "hhsize", "auto_ownership"]].head(10))


Sample of unexpectedly affected households (home_zone_id, income, hhsize, auto_ownership):


,home_zone_id,income,hhsize,auto_ownership
household_id,,,,
467,611,36941,5,2
1583,1380,0,3,1
2612,1978,155088,8,2
5308,5702,66235,6,3
6008,5702,61065,6,2
6538,5828,129240,5,2
8804,5878,64620,4,2
16017,9910,34464,5,2
31864,16297,173289,7,2
